# Garim Visual PII OCR Pipeline

Scene-aware adaptive OCR baseline for visual PII. Results store masked text, time intervals, original-video bbox coordinates, confidence, and source frame numbers for later blur or mosaic rendering.

The default install works on a Colab CPU runtime. Select a GPU runtime and install the matching Paddle GPU wheel when processing larger videos.

In [ ]:
# 1. Install dependencies
!pip install -q paddlepaddle paddleocr "scenedetect[opencv]" pandas

import platform
print("Python runtime:", platform.python_version())
try:
    import paddle
    print("Paddle device:", paddle.device.get_device())
except Exception as exc:
    print("Paddle import failed:", exc)

## Upload module and video

Upload `garim_visual_pii_ocr_pipeline.py` from this directory and one test video. Do not commit uploaded videos or generated output.

In [ ]:
# 2. Upload the reusable module
from google.colab import files

uploaded = files.upload()
if "garim_visual_pii_ocr_pipeline.py" not in uploaded:
    raise RuntimeError("Upload garim_visual_pii_ocr_pipeline.py first.")

In [ ]:
# 3. Configure sampling and output
from pathlib import Path
from garim_visual_pii_ocr_pipeline import VisualPIIConfig, run_visual_pii_pipeline

UPLOAD_ID = "colab-manual-test"
OUTPUT_DIR = Path("/content/garim_visual_pii_output")
CONFIG = VisualPIIConfig(
    base_fps=2.0,
    boundary_fps=5.0,
    burst_fps=8.0,
    recognition_batch_size=16,
    boundary_radius_sec=0.5,
    interval_padding_sec=0.4,
)
print(CONFIG)

In [ ]:
# 4. Upload a video
video_upload = files.upload()
video_names = [name for name in video_upload if name.lower().endswith((".mp4", ".mov", ".mkv", ".avi", ".webm"))]
if not video_names:
    raise RuntimeError("Upload one video file.")
VIDEO_PATH = Path(video_names[0])
print("Video:", VIDEO_PATH)

In [ ]:
# 5. Run scene detection, adaptive sampling, OCR, and interval merge
result = run_visual_pii_pipeline(
    video_path=VIDEO_PATH,
    upload_id=UPLOAD_ID,
    output_dir=OUTPUT_DIR,
    config=CONFIG,
)
summary = {key: value for key, value in result.items() if key not in {"detections", "review_thumbnails"}}
summary["detection_count"] = len(result["detections"])
summary

In [ ]:
# 6. Review masked detections only
import json
print(json.dumps(result["detections"], ensure_ascii=False, indent=2))

In [ ]:
# 7. Download JSON, CSV, and blurred review thumbnails
import shutil
zip_path = shutil.make_archive("/content/garim_visual_pii_result", "zip", OUTPUT_DIR)
files.download(zip_path)

In [ ]:
# 8. Print generated JSON and CSV results
import json

json_path = OUTPUT_DIR / "visual_pii_detections.json"
csv_path = OUTPUT_DIR / "visual_pii_detections.csv"

print("=== JSON ===")
print(json_path.read_text(encoding="utf-8"))
print("\n=== CSV ===")
print(csv_path.read_text(encoding="utf-8"))

In [ ]:
# 9. Preview and save frames immediately before OCR
import cv2
import matplotlib.pyplot as plt

from garim_visual_pii_ocr_pipeline import (
    get_video_meta,
    detect_scenes,
    find_motion_timestamps,
    build_candidate_timestamps,
    read_unique_frames,
)

meta = get_video_meta(VIDEO_PATH)
scenes = detect_scenes(VIDEO_PATH, meta, CONFIG.scene_threshold)
motion = find_motion_timestamps(VIDEO_PATH, meta, CONFIG)
timestamps = build_candidate_timestamps(meta, scenes, CONFIG, motion)
frames = read_unique_frames(VIDEO_PATH, meta, timestamps, CONFIG)

print(f"영상: {VIDEO_PATH}")
print(f"영상 길이: {meta.duration_sec:.1f}초")
print(f"장면 수: {len(scenes)}")
print(f"OCR 전달 프레임 수: {len(frames)}")

limit = min(40, len(frames))
cols = 4
rows = (limit + cols - 1) // cols
if limit:
    plt.figure(figsize=(16, rows * 3))
    for i, frame in enumerate(frames[:limit]):
        image = cv2.cvtColor(frame.image, cv2.COLOR_BGR2RGB)
        plt.subplot(rows, cols, i + 1)
        plt.imshow(image)
        plt.title(f"Frame {frame.frame_no}\n{frame.timestamp_sec:.2f}s")
        plt.axis("off")
    plt.tight_layout()
    plt.show()

frame_output = Path("/content/ocr_input_frames")
frame_output.mkdir(exist_ok=True)
for frame in frames:
    path = frame_output / f"frame_{frame.frame_no:06d}_{frame.timestamp_sec:.2f}s.jpg"
    cv2.imwrite(str(path), frame.image)

print(f"저장 완료: {frame_output}")